# Modulo 7 — AI Feedback

Dopo ogni backtest: **analizza gli errori**, **propone miglioramenti**, **ottimizza** i parametri di rischio, crea una **nuova versione** della strategia e mantiene lo **storico delle versioni**.

L'ottimizzazione è volutamente conservativa (griglia grossolana + obiettivo che penalizza il drawdown) per non inseguire il rumore.

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import DataEngine, generate_ohlcv
from trading_ai.feature_engineering import FeatureEngine
from trading_ai.pattern_discovery.clustering import FeatureClusterer
from trading_ai.strategy_generator import Strategy, RiskParams
from trading_ai.feedback import FeedbackEngine

In [ ]:
eng = DataEngine()
h1 = eng.to_timeframe(eng.load_dataframe(generate_ohlcv(n=120_000, seed=5)), 'H1')
feats = FeatureEngine().transform(h1, groups=['indicator', 'volatility'], dropna=True)
cols = [c for c in feats.columns if c not in ('open','high','low','close','volume')]
clu = FeatureClusterer(n_clusters=8, feature_columns=cols).fit(feats)
strat = Strategy(name='PAT01_LONG', cluster_id=1, direction=1, clusterer=clu,
                 risk=RiskParams(sl_atr=2, tp_atr=3, max_bars=30))

## Ciclo di feedback

Diagnosi degli errori + ottimizzazione automatica + nuova versione.

In [ ]:
fb = FeedbackEngine(min_trades=20)
res = fb.improve(strat, feats)
print('PROBLEMI:'); [print(' -', x) for x in res.diagnosis['issues']]
print('SUGGERIMENTI:'); [print(' -', x) for x in res.diagnosis['suggestions']]
print()
for v in res.versions:
    print(f"v{v.version} (parent={v.parent}) ret={v.metrics['total_return']:.4f} "
          f"dd={v.metrics['max_drawdown']:.4f} risk={v.risk['sl_atr']}/{v.risk['tp_atr']}")

In [ ]:
# Storico versioni persistito su file (versionabile in git).
import os
out = Path(os.environ.get('TRADING_AI_ROOT', '.')) / 'strategies' / 'PAT01_LONG_history.json'
print('Salvato:', fb.save_history(out))
res.optimization_table.head()

## ✅ Modulo 7 verificato

Analisi automatica, ottimizzazione robusta, versioning con storico persistente. Test: `pytest tests/test_feedback.py` (7/7).